In [1]:
import pandas as pd
import os
from dotenv import load_dotenv
from mistralai.client import Mistral
import numpy as np

from src.config import DATA_DIR, RESULT_DIR

load_dotenv()

client = Mistral(api_key=os.getenv("MISTRAL_API_KEY"))

local_dir = RESULT_DIR / "implicite_segmentation"
local_dir.mkdir(parents=True, exist_ok=True)



LABEL_CATS = ["Question", "Neither", "Assertion"]
SCORE_CATS = [1.0, 2.0, 3.0, 4.0, 5.0]
SUBJ_IMPUTE = 0.0


def block_to_sequence(block: pd.DataFrame, subj_mean = 0., subj_std = 0., order_col: str | None = None) -> np.ndarray:
    if order_col is not None:
        block = block.sort_values(order_col)

    cat_idx = block["rule_based_SA-label"].map({c: i for i, c in enumerate(LABEL_CATS)})
    onehot = np.eye(len(LABEL_CATS))[cat_idx.to_numpy()]

    raw = block["raw_prediction_1_5"].to_numpy(dtype=float)   # NaN où absent
    valid = (~np.isnan(raw)).astype(float)  # 1.0 si défini, 0.0 sinon

    #value = np.where(np.isnan(raw), SUBJ_IMPUTE, raw)         # impute la constante
    value = np.where(np.isnan(raw), 0.0, (raw - subj_mean) / subj_std)

    # Concaténation colonne par colonne -> (L, 5)
    return np.column_stack([onehot, value, valid])


import torch
from torch.nn.utils.rnn import pad_sequence

def df_to_padded_batch(df, label, subj_mean=None, subj_std=None, order_col=None):
    """Retourne (padded, lengths, labels, ids).
       padded  : (N_blocs, L_max, 5) float32
       lengths : (N_blocs,) int64  -- longueurs réelles, pour pack_padded_sequence
       labels  : (N_blocs,) int64  -- constante = label
       ids     : ordre des review-id (traçabilité)
    """
    seqs, lengths, ids = [], [], []
    for rid, block in df.groupby("review-id", sort=False):
        s = block_to_sequence(block, subj_mean, subj_std, order_col=order_col)
        seqs.append(torch.tensor(s, dtype=torch.float32))
        lengths.append(len(s))
        ids.append(rid)
    # pad_sequence avec batch_first : (N, L_max, 5), padding=0.0
    padded = pad_sequence(seqs, batch_first=True, padding_value=0.0)
    lengths = torch.tensor(lengths, dtype=torch.int64)
    labels = torch.full((len(seqs),), label, dtype=torch.int64)
    return padded, lengths, labels, np.array(ids)

/home/cyaiche/generated_text_detection_hub/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-07-01 10:36:34.873 | INFO     | src.config:<module>:12 - PROJ_ROOT path is: /home/cyaiche/generated_text_detection_hub


## Récupération de la base d'entrainement

In [2]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import os
output_file_path_sentence = "sentences_degrees_calibration.csv"
df = pd.read_csv(local_dir / output_file_path_sentence, encoding="utf-8-sig")


In [ ]:
df

In [ ]:
hwt_df = df[df["label"] == 0]
mgt_df = df[df["label"] == 1]


In [ ]:
hwt_df

In [ ]:
print(len(df))
print(len(hwt_df['review-id'].unique()))
print(len(mgt_df['review-id'].unique()))
print(len(df['review-id'].unique()))
df

## Récupération de la base de test

In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import os

output_file_path_sentence = "sentences_degrees_test.csv"
df_test = pd.read_csv(local_dir / output_file_path_sentence, encoding="utf-8-sig")

hwt_df_test = df_test[df_test["label"] == 0]
mgt_df_test = df_test[df_test["label"] == 1]


In [ ]:
print(len(df_test))
print(len(hwt_df_test['review-id'].unique()))
print(len(mgt_df_test['review-id'].unique()))
print(len(df_test['review-id'].unique()))
df_test

# Création des Features

In [ ]:
def build_dataset(hwt, mgt, subj_mean, subj_std, order_col=None):
    """Assemble HWT(0)+MGT(1) en (padded, lengths, labels, ids) avec un L_max GLOBAL."""
    seqs, lengths, labels, ids = [], [], [], []
    for df, lab in [(hwt, 0), (mgt, 1)]:
        for rid, block in df.groupby("review-id", sort=False):
            s = block_to_sequence(block, subj_mean, subj_std, order_col)
            seqs.append(torch.tensor(s, dtype=torch.float32))
            lengths.append(len(s)); labels.append(lab); ids.append(rid)
    padded = pad_sequence(seqs, batch_first=True, padding_value=0.0)  # L_max global
    return (padded,
            torch.tensor(lengths, dtype=torch.int64),
            torch.tensor(labels, dtype=torch.int64),
            np.array(ids))

In [ ]:
train_all = pd.concat([hwt_df, mgt_df])
subj_mean = train_all["raw_prediction_1_5"].mean()
subj_std  = train_all["raw_prediction_1_5"].std()

# train
Xtr, Ltr, Ytr, _ = build_dataset(hwt_df, mgt_df, subj_mean, subj_std)
# test externe, MÊMES stats
Xte, Lte, Yte, _ = build_dataset(hwt_df_test, mgt_df_test, subj_mean, subj_std)

from torch.utils.data import TensorDataset, DataLoader
train_dl = DataLoader(TensorDataset(Xtr, Ltr, Ytr), batch_size=64, shuffle=True)
test_dl  = DataLoader(TensorDataset(Xte, Lte, Yte), batch_size=64, shuffle=False)

# Entrainement

In [ ]:
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence

class SimpleGRU(nn.Module):
    """GRU naïf : 1 couche, unidirectionnel, dernier état caché -> classif binaire."""
    def __init__(self, n_features=5, hidden=32):
        super().__init__()
        self.gru = nn.GRU(input_size=n_features, hidden_size=hidden,
                          num_layers=1, batch_first=True)
        self.head = nn.Linear(hidden, 1)   # 1 logit -> BCEWithLogits

    def forward(self, x, lengths):
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True,
                                      enforce_sorted=False)
        _, h_n = self.gru(packed)
        last = h_n[-1]                      # (batch, hidden)
        return self.head(last).squeeze(-1) # (batch,) logits

In [ ]:
import numpy as np, torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

def train_model(model, train_dl, epochs=15, lr=1e-3, device="cpu"):
    model.to(device)
    crit = nn.BCEWithLogitsLoss()                    # équilibre -> pas de pos_weight
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for ep in range(epochs):
        model.train(); tot = 0.0
        for xb, lb, yb in train_dl:
            xb, yb = xb.to(device), yb.float().to(device)
            opt.zero_grad()
            loss = crit(model(xb, lb), yb)
            loss.backward(); opt.step()
            tot += loss.item() * len(yb)
        if ep % 3 == 0 or ep == epochs-1:
            print(f"epoch {ep:2d}  loss {tot/len(train_dl.dataset):.4f}")

@torch.no_grad()
def evaluate(model, test_dl, device="cpu"):
    model.eval(); ys, ps = [], []
    for xb, lb, yb in test_dl:
        proba = torch.sigmoid(model(xb.to(device), lb)).cpu().numpy()
        ps.append(proba); ys.append(yb.numpy())
    y = np.concatenate(ys); p = np.concatenate(ps); pred = (p >= 0.5).astype(int)
    print("\nconfusion [[TN FP][FN TP]] :\n", confusion_matrix(y, pred))
    print(classification_report(y, pred, digits=3))
    print("AUC :", round(roc_auc_score(y, p), 4))
    return y, p          # <-- pour la ROC

In [ ]:
model = SimpleGRU()
train_model(model, train_dl, epochs=15)   # train_dl = ton DataLoader train réel
evaluate(model, test_dl)                   # test_dl = ton test externe

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay

y_te, p_te = evaluate(model, test_dl)   # récupère labels et probabilités du test
result_file = f"gru_result_1.npz"
np.savez(local_dir / f"{result_file}", y_true=y_te, y_proba=p_te)


fig, ax = plt.subplots(figsize=(6, 6))
RocCurveDisplay.from_predictions(y_te, p_te, ax=ax, name="GRU baseline")
ax.plot([0, 1], [0, 1], linestyle="--", color="grey", label="aléatoire")
ax.set_xlabel("Taux de faux positifs (FPR)")
ax.set_ylabel("Taux de vrais positifs (TPR)")
ax.set_title("Courbe ROC — GRU baseline")
ax.legend(); plt.tight_layout()
#plt.savefig("roc_gru.png", dpi=150);
plt.show()